## 1. Tooth Detection and Segmentation
Process dental cases with the Tooth Segmentation + Recognition model.
This step outputs bounding-box images, mask overlays, and per-case JSON files.

Next step output used by Step 2:
- segmentation+recognition-dataset

In [ ]:
# # Step 1: Tooth Detection and Segmentation on raw_data (all cases)
# import os
# import json
# import random
# from pathlib import Path

# import cv2
# import numpy as np
# from tqdm.auto import tqdm

# # IMPORTANT for Windows/Jupyter: use non-interactive backend to avoid Tkinter issues
# os.environ["MPLBACKEND"] = "Agg"
# import matplotlib
# matplotlib.use("Agg")
# import matplotlib.pyplot as plt

# from ultralytics import YOLO
# from detectron2.engine import DefaultPredictor
# from detectron2.config import get_cfg
# from detectron2 import model_zoo


# def compute_iou(boxA, boxB):
#     xA = max(boxA[0], boxB[0])
#     yA = max(boxA[1], boxB[1])
#     xB = min(boxA[2], boxB[2])
#     yB = min(boxA[3], boxB[3])
#     inter = max(0, xB - xA) * max(0, yB - yA)
#     areaA = max(0, boxA[2] - boxA[0]) * max(0, boxA[3] - boxA[1])
#     areaB = max(0, boxB[2] - boxB[0]) * max(0, boxB[3] - boxB[1])
#     union = float(areaA + areaB - inter)
#     return inter / union if union > 0 else 0.0


# def get_device():
#     import torch

#     if torch.cuda.is_available():
#         device = "cuda"
#         print(f"Using device: cuda | GPU: {torch.cuda.get_device_name(0)}")
#     elif hasattr(torch.backends, "mps") and torch.backends.mps.is_available():
#         device = "mps"
#         print("Using device: mps")
#     else:
#         device = "cpu"
#         print("Using device: cpu")
#     return device


# def get_detectron2_device(device: str) -> str:
#     # Detectron2 reliably supports cuda/cpu. If main device is mps, use cpu for Detectron2.
#     if device == "cuda":
#         return "cuda"
#     if device == "mps":
#         print("Detectron2 uses cpu because mps support is limited.")
#         return "cpu"
#     return "cpu"


# def initialize_models(raw_data_dir: Path):
#     device = get_device()
#     detectron_device = get_detectron2_device(device)

#     # Try common locations for model weights (supports both 'weights' and 'weights(model)')
#     yolo_candidates = [
#         raw_data_dir / "Tooth Segmentation + Recognition model" / "weights(model)" / "Tooth_seg_pano_20250319.pt",
#         raw_data_dir / "Tooth Segmentation + Recognition model" / "weights" / "Tooth_seg_pano_20250319.pt",
#         Path("Tooth Segmentation + Recognition model") / "weights(model)" / "Tooth_seg_pano_20250319.pt",
#         Path("Tooth Segmentation + Recognition model") / "weights" / "Tooth_seg_pano_20250319.pt",
#     ]
#     detectron_candidates = [
#         raw_data_dir / "Tooth Segmentation + Recognition model" / "weights(model)" / "Tooth_seg_crop_20250424.pth",
#         raw_data_dir / "Tooth Segmentation + Recognition model" / "weights" / "Tooth_seg_crop_20250424.pth",
#         Path("Tooth Segmentation + Recognition model") / "weights(model)" / "Tooth_seg_crop_20250424.pth",
#         Path("Tooth Segmentation + Recognition model") / "weights" / "Tooth_seg_crop_20250424.pth",
#     ]

#     yolo_path = next((p for p in yolo_candidates if p.exists()), None)
#     detectron_path = next((p for p in detectron_candidates if p.exists()), None)

#     if yolo_path is None or detectron_path is None:
#         checked = [str(p) for p in (yolo_candidates + detectron_candidates)]
#         raise FileNotFoundError(
#             "Model weights not found. Checked paths:\n" + "\n".join(checked)
#         )

#     print(f"YOLO weights: {yolo_path}")
#     print(f"Detectron2 weights: {detectron_path}")

#     yolo_model = YOLO(str(yolo_path))
#     yolo_model.to(device)

#     cfg = get_cfg()
#     cfg.merge_from_file(model_zoo.get_config_file("COCO-InstanceSegmentation/mask_rcnn_R_50_FPN_3x.yaml"))
#     cfg.MODEL.ROI_HEADS.NUM_CLASSES = 1
#     cfg.MODEL.WEIGHTS = str(detectron_path)
#     cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = 0.5
#     cfg.MODEL.DEVICE = detectron_device
#     detectron_predictor = DefaultPredictor(cfg)

#     print("Models initialized successfully")
#     return yolo_model, detectron_predictor


# def process_panoramic_image(pano_img_rgb, yolo_model):
#     results = yolo_model(pano_img_rgb, verbose=False)
#     h, w = pano_img_rgb.shape[:2]
#     all_masks = []

#     for result in results:
#         if result.masks is None or not hasattr(result.masks, "data"):
#             continue

#         temp = []
#         num_masks = len(result.masks.xy) if hasattr(result.masks, "xy") else 0

#         for i in range(num_masks):
#             mask_matrix = result.masks.data[i].cpu().numpy()
#             resized_mask = cv2.resize(
#                 mask_matrix.astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST
#             ).astype(bool)

#             bbox = result.boxes.xyxy[i].cpu().numpy().tolist()
#             cls_id = int(result.boxes.cls[i].item())

#             temp.append({
#                 "id": i,
#                 "matrix_resized": resized_mask,
#                 "confidence": float(result.boxes.conf[i].item()),
#                 "class_id": cls_id,
#                 "class_name": str(result.names[cls_id]),
#                 "bbox": bbox,
#             })

#         # Remove duplicates by IoU
#         kept = []
#         for m in temp:
#             duplicated = False
#             for k in kept:
#                 if compute_iou(m["bbox"], k["bbox"]) > 0.5:
#                     if m["confidence"] > k["confidence"]:
#                         kept.remove(k)
#                         kept.append(m)
#                     duplicated = True
#                     break
#             if not duplicated:
#                 kept.append(m)

#         all_masks.extend(kept)

#     return all_masks


# def crop_and_segment_teeth(pano_img_rgb, mask_data_list, detectron_predictor, pad=20):
#     outputs = []
#     for data in mask_data_list:
#         mask = data["matrix_resized"].astype(np.uint8) * 255
#         x, y, w, h = cv2.boundingRect(mask)
#         x1 = max(x - pad, 0)
#         y1 = max(y - pad, 0)
#         x2 = min(x + w + pad, pano_img_rgb.shape[1])
#         y2 = min(y + h + pad, pano_img_rgb.shape[0])

#         cropped = pano_img_rgb[y1:y2, x1:x2]
#         pred = detectron_predictor(cropped)
#         instances = pred["instances"].to("cpu")

#         segments = []
#         all_pixels = []
#         if len(instances) > 0:
#             masks = instances.pred_masks.numpy()
#             scores = instances.scores.numpy()
#             for idx, (seg_mask, score) in enumerate(zip(masks, scores)):
#                 rows, cols = np.where(seg_mask)
#                 pixel_coords = [(int(x1 + c), int(y1 + r)) for r, c in zip(rows, cols)]
#                 all_pixels.extend(pixel_coords)
#                 segments.append({
#                     "mask_id": idx,
#                     "score": float(score),
#                     "num_pixels": len(pixel_coords),
#                     "pixel_coordinates": pixel_coords,
#                 })

#         outputs.append({
#             "tooth_id": data["class_name"],
#             "crop_coords": [int(x1), int(y1), int(x2), int(y2)],
#             "num_segments": len(segments),
#             "segments": segments,
#             "pixel_coordinates": all_pixels,
#             "total_pixels": len(all_pixels),
#             "detectron_output": pred,
#         })

#     return outputs


# def save_bbox_visualization(pano_img_rgb, list_of_masks, output_path):
#     vis = pano_img_rgb.copy()
#     for m in list_of_masks:
#         x1, y1, x2, y2 = map(int, m["bbox"])
#         label = str(m["class_name"])
#         cv2.rectangle(vis, (x1, y1), (x2, y2), (0, 255, 0), 2)
#         cv2.putText(vis, label, (x1, max(20, y1 - 8)), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 255, 0), 2)

#     fig = plt.figure(figsize=(12, 8))
#     plt.imshow(vis)
#     plt.axis("off")
#     plt.title("Bounding Boxes with Tooth ID")
#     plt.savefig(output_path, bbox_inches="tight", dpi=150)
#     plt.close(fig)


# def save_mask_overlay(pano_img_rgb, list_seg_tooth, output_path, alpha=0.5):
#     vis = pano_img_rgb.copy()

#     for result in list_seg_tooth:
#         x1, y1, x2, y2 = result["crop_coords"]
#         pred = result["detectron_output"]
#         instances = pred["instances"].to("cpu")

#         if len(instances) == 0:
#             continue

#         masks = instances.pred_masks.numpy()
#         for mask in masks:
#             h, w = mask.shape
#             color = np.array([random.randint(0, 255) for _ in range(3)], dtype=np.uint8)

#             color_mask = np.zeros((h, w, 3), dtype=np.uint8)
#             color_mask[mask] = color

#             crop = vis[y1:y2, x1:x2]
#             blended = np.where(
#                 mask[:, :, None],
#                 cv2.addWeighted(crop, 1 - alpha, color_mask, alpha, 0),
#                 crop,
#             )
#             vis[y1:y2, x1:x2] = blended

#     fig = plt.figure(figsize=(16, 8))
#     plt.imshow(vis)
#     plt.axis("off")
#     plt.title("Mask Overlay")
#     plt.savefig(output_path, bbox_inches="tight", dpi=150)
#     plt.close(fig)


# def save_results_json(case_num, list_of_masks, list_seg_tooth, output_dir):
#     result = {
#         "case_number": str(case_num),
#         "num_teeth_detected": len(list_of_masks),
#         "teeth_data": [],
#     }

#     for mask_data, seg_data in zip(list_of_masks, list_seg_tooth):
#         result["teeth_data"].append({
#             "tooth_id": mask_data["class_name"],
#             "confidence": float(mask_data["confidence"]),
#             "bbox": [float(v) for v in mask_data["bbox"]],
#             "crop_coords": seg_data["crop_coords"],
#             "num_segments": seg_data["num_segments"],
#             "total_pixels": seg_data["total_pixels"],
#             "pixel_coordinates": seg_data["pixel_coordinates"],
#             "segments_detail": seg_data["segments"],
#         })

#     with open(output_dir / f"case_{case_num}_results.json", "w", encoding="utf-8") as f:
#         json.dump(result, f, ensure_ascii=False, indent=2)


# def process_case(case_dir, yolo_model, detectron_predictor, output_root):
#     case_num = case_dir.name.replace("case ", "")
#     png_path = case_dir / f"case_{case_num}.png"
#     if not png_path.exists():
#         png_files = sorted(case_dir.glob("*.png"))
#         if not png_files:
#             return False, "No PNG found"
#         png_path = png_files[0]

#     img_bgr = cv2.imread(str(png_path))
#     if img_bgr is None:
#         return False, "Image read failed"

#     img_rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
#     list_of_masks = process_panoramic_image(img_rgb, yolo_model)
#     if len(list_of_masks) == 0:
#         return False, "No teeth detected"

#     list_seg_tooth = crop_and_segment_teeth(img_rgb, list_of_masks, detectron_predictor)

#     case_out = output_root / f"case {case_num}"
#     case_out.mkdir(parents=True, exist_ok=True)

#     save_bbox_visualization(img_rgb, list_of_masks, case_out / f"case_{case_num}_bounding_boxes.png")
#     save_mask_overlay(img_rgb, list_seg_tooth, case_out / f"case_{case_num}_mask_overlay.png")
#     save_results_json(case_num, list_of_masks, list_seg_tooth, case_out)

#     return True, f"OK ({len(list_of_masks)} teeth)"


# def run_inference_on_raw_dataset(max_cases=None):
#     project_root = Path.cwd()
#     # Adjusting paths relative to the current working directory
#     raw_data_dir = project_root.parent / "data"
#     cases_root = raw_data_dir / "500 cases with annotation"

#     if not cases_root.exists():
#         raise FileNotFoundError(f"Cases folder not found: {cases_root}")

#     output_root = project_root / "segmentation+recognition-dataset"
#     output_root.mkdir(parents=True, exist_ok=True)

#     yolo_model, detectron_predictor = initialize_models(raw_data_dir)

#     case_dirs = sorted(
#         [d for d in cases_root.iterdir() if d.is_dir() and d.name.startswith("case ")],
#         key=lambda p: int(p.name.replace("case ", "")),
#     )

#     if max_cases is not None:
#         case_dirs = case_dirs[:max_cases]
#         print(f"Limiting execution to {max_cases} sample cases.")

#     ok_count = 0
#     fail_count = 0
#     print(f"Processing cases: {len(case_dirs)}")

#     for case_dir in tqdm(case_dirs, desc="raw_data cases", unit="case"):
#         try:
#             ok, msg = process_case(case_dir, yolo_model, detectron_predictor, output_root)
#             if ok:
#                 ok_count += 1
#             else:
#                 fail_count += 1
#         except Exception as e:
#             fail_count += 1
#             print(f"Error in {case_dir.name}: {e}")

#     summary = {
#         "source": "data/500 cases with annotation-raw",
#         "processed": ok_count,
#         "failed": fail_count,
#         "total": len(case_dirs),
#     }

#     with open(output_root / "summary.json", "w", encoding="utf-8") as f:
#         json.dump(summary, f, indent=2)

#     print("\nDone")
#     print(f"Output folder: {output_root.resolve()}")
#     print(json.dumps(summary, indent=2))


# # RUN ON ALL CASES
# run_inference_on_raw_dataset()

# # Uncomment the line below to run on ALL cases
# # run_inference_on_raw_dataset()

d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\detectron2\model_zoo\model_zoo.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources


Using device: cuda | GPU: NVIDIA GeForce RTX 4080 Laptop GPU
YOLO weights: c:\Users\jaopi\Desktop\SP\data\Tooth Segmentation + Recognition model\weights\Tooth_seg_pano_20250319.pt
Detectron2 weights: c:\Users\jaopi\Desktop\SP\data\Tooth Segmentation + Recognition model\weights\Tooth_seg_crop_20250424.pth


d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\fvcore\common\checkpoint.py:252: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  return torch.load(f, map_location=t

Models initialized successfully
Processing cases: 500


raw_data cases:   0%|          | 0/500 [00:00<?, ?case/s]d:\Users\jaopi\anaconda3\envs\sp_project\lib\site-packages\torch\functional.py:534: UserWarning: torch.meshgrid: in an upcoming release, it will be required to pass the indexing argument. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\TensorShape.cpp:3596.)
  return _VF.meshgrid(tensors, **kwargs)  # type: ignore[attr-defined]
raw_data cases:   3%|▎         | 13/500 [01:41<1:03:18,  7.80s/case]


KeyboardInterrupt: 

## 2. Map tooth positions to caries regions (JSON output)
Combined script that processes dental cases to:
1. Map tooth positions to caries regions (JSON output)
2. Generate visual alignment debug images (PNG output)

Data Sources:
- JSON: segmentation+recognition-dataset/case X/case_X_results.json
  Contains pixel_coordinates (list of [x, y]) for each tooth
- ROI Image: raw_data/500-roi/case_X.png
  Binary mask where non-zero = caries, 0 = normal

Output (single root folder):
- caries_mapping_output/case X/case_X_caries_mapping.json
- caries_mapping_output/case X/case_X_alignment_detailed.png
- caries_mapping_output/caries_mapping_results.csv (summary)

In [ ]:
# import cv2
# import json
# import numpy as np
# import pandas as pd
# from pathlib import Path
# from tqdm.auto import tqdm


# # =============================================================================
# # ROI & JSON Loading Functions
# # =============================================================================

# def load_roi_image(roi_path: Path):
#     """Load ROI mask and normalize to 2D grayscale array."""
#     roi_img = cv2.imread(str(roi_path), cv2.IMREAD_UNCHANGED)
#     if roi_img is None:
#         raise FileNotFoundError(f"Could not load ROI image: {roi_path}")

#     # Some PNGs are loaded as (H, W, 1); convert to strict 2D for downstream ops.
#     if roi_img.ndim == 3:
#         if roi_img.shape[2] == 1:
#             roi_img = roi_img[:, :, 0]
#         else:
#             roi_img = cv2.cvtColor(roi_img, cv2.COLOR_BGR2GRAY)

#     return roi_img


# def load_tooth_json(json_path: Path):
#     """Load tooth segmentation JSON with pixel coordinates."""
#     with open(json_path, "r", encoding="utf-8") as f:
#         return json.load(f)


# # =============================================================================
# # Caries Mapping Functions
# # =============================================================================

# def calculate_caries_overlap(roi_img, pixel_coordinates):
#     """Calculate overlap between tooth pixels and caries region in ROI."""
#     if not pixel_coordinates:
#         return 0, 0, 0.0, []

#     height, width = roi_img.shape[:2]
#     caries_count = 0
#     caries_coords = []
#     valid_pixels = 0

#     for coord in pixel_coordinates:
#         x, y = coord[0], coord[1]

#         if 0 <= x < width and 0 <= y < height:
#             valid_pixels += 1
#             if roi_img[y, x] > 0:
#                 caries_count += 1
#                 caries_coords.append([x, y])

#     percentage = (caries_count / valid_pixels * 100) if valid_pixels > 0 else 0.0
#     return caries_count, valid_pixels, percentage, caries_coords


# def generate_caries_mapping(roi_img, tooth_data, case_num):
#     """Generate per-tooth caries mapping results for one case."""
#     results = []

#     for tooth in tooth_data.get("teeth_data", []):
#         tooth_id = tooth.get("tooth_id", "unknown")
#         pixel_coords = tooth.get("pixel_coordinates", [])
#         confidence = tooth.get("confidence", 0.0)

#         caries_count, total_pixels, percentage, caries_coords = calculate_caries_overlap(
#             roi_img, pixel_coords
#         )

#         results.append(
#             {
#                 "case_number": int(case_num),
#                 "tooth_id": tooth_id,
#                 "confidence": float(confidence),
#                 "total_pixels": int(total_pixels),
#                 "caries_pixels": int(caries_count),
#                 "caries_percentage": round(float(percentage), 4),
#                 "has_caries": bool(caries_count > 0),
#                 "caries_coordinates": caries_coords,
#             }
#         )

#     return results


# # =============================================================================
# # Visual Alignment Functions
# # =============================================================================

# def create_alignment_visualization(roi_img, tooth_data, alpha=0.5):
#     """Create blended visualization: ROI background + tooth pixels overlay."""
#     height, width = roi_img.shape[:2]

#     roi_normalized = np.where(roi_img > 0, 255, 0).astype(np.uint8)
#     roi_bgr = cv2.cvtColor(roi_normalized, cv2.COLOR_GRAY2BGR)
#     tooth_overlay = np.zeros((height, width, 3), dtype=np.uint8)

#     colors = [
#         (0, 255, 0),
#         (0, 255, 255),
#         (255, 255, 0),
#         (0, 165, 255),
#         (255, 0, 255),
#         (128, 255, 0),
#         (255, 128, 0),
#         (0, 128, 255),
#     ]

#     tooth_count = 0
#     total_pixels_drawn = 0
#     out_of_bounds = 0

#     for tooth in tooth_data.get("teeth_data", []):
#         pixel_coords = tooth.get("pixel_coordinates", [])
#         color = colors[tooth_count % len(colors)]
#         tooth_count += 1

#         for coord in pixel_coords:
#             x, y = coord[0], coord[1]
#             if 0 <= x < width and 0 <= y < height:
#                 tooth_overlay[y, x] = color
#                 total_pixels_drawn += 1
#             else:
#                 out_of_bounds += 1

#     tooth_mask = np.any(tooth_overlay > 0, axis=2)
#     result = roi_bgr.copy()
#     blended = cv2.addWeighted(roi_bgr, 1 - alpha, tooth_overlay, alpha, 0)
#     result[tooth_mask] = blended[tooth_mask]
#     strong_overlay = cv2.addWeighted(result, 0.7, tooth_overlay, 0.3, 0)
#     result[tooth_mask] = strong_overlay[tooth_mask]

#     return result, {
#         "teeth_count": tooth_count,
#         "pixels_drawn": total_pixels_drawn,
#         "out_of_bounds": out_of_bounds,
#         "roi_shape": (height, width),
#     }


# def create_detailed_alignment_image(roi_img, tooth_data):
#     """Create 3-panel alignment image for QA/debug."""
#     height, width = roi_img.shape[:2]
#     roi_normalized = np.where(roi_img > 0, 255, 0).astype(np.uint8)

#     roi_colored = cv2.cvtColor(roi_normalized, cv2.COLOR_GRAY2BGR)
#     caries_mask = roi_img > 0
#     roi_colored[caries_mask] = [0, 0, 255]

#     tooth_only = np.zeros((height, width, 3), dtype=np.uint8)
#     for tooth in tooth_data.get("teeth_data", []):
#         for coord in tooth.get("pixel_coordinates", []):
#             x, y = coord[0], coord[1]
#             if 0 <= x < width and 0 <= y < height:
#                 tooth_only[y, x] = [0, 255, 0]

#     blended, stats = create_alignment_visualization(roi_img, tooth_data, alpha=0.6)

#     tooth_mask = np.any(tooth_only > 0, axis=2)
#     overlap_pixels = int(np.sum(tooth_mask & caries_mask))
#     caries_pixels = int(np.sum(caries_mask))
#     tooth_pixels = int(np.sum(tooth_mask))

#     font = cv2.FONT_HERSHEY_SIMPLEX
#     cv2.putText(roi_colored, "ROI (Red=Caries)", (20, 60), font, 1.5, (255, 255, 255), 3)
#     cv2.putText(tooth_only, "Teeth (Green)", (20, 60), font, 1.5, (255, 255, 255), 3)
#     cv2.putText(blended, "ALIGNMENT CHECK", (20, 60), font, 1.5, (0, 255, 255), 3)

#     stats_text = [
#         f"Teeth: {stats['teeth_count']}",
#         f"Tooth pixels: {tooth_pixels:,}",
#         f"Caries pixels: {caries_pixels:,}",
#         f"Overlap: {overlap_pixels:,}",
#     ]
#     y_offset = 120
#     for text in stats_text:
#         cv2.putText(blended, text, (20, y_offset), font, 1.0, (255, 255, 0), 2)
#         y_offset += 40

#     scale = 0.5
#     roi_small = cv2.resize(roi_colored, None, fx=scale, fy=scale)
#     tooth_small = cv2.resize(tooth_only, None, fx=scale, fy=scale)
#     blended_small = cv2.resize(blended, None, fx=scale, fy=scale)
#     combined = np.hstack([roi_small, tooth_small, blended_small])

#     overlap_stats = {
#         "overlap_pixels": overlap_pixels,
#         "caries_pixels": caries_pixels,
#         "tooth_pixels": tooth_pixels,
#     }
#     return combined, stats, overlap_stats


# # =============================================================================
# # Pipeline Functions for Current Project Structure
# # =============================================================================

# def parse_case_num_from_json(json_path: Path):
#     """Extract numeric case id from filename: case_123_results.json -> 123."""
#     stem = json_path.stem
#     parts = stem.split("_")
#     if len(parts) < 3:
#         raise ValueError(f"Unexpected JSON filename format: {json_path.name}")
#     return int(parts[1])


# def collect_case_records(segmentation_root: Path, roi_root: Path):
#     """Collect case records from Step 1 output and matching ROI paths."""
#     records = []

#     case_dirs = sorted(
#         [p for p in segmentation_root.iterdir() if p.is_dir() and p.name.startswith("case ")],
#         key=lambda p: int(p.name.replace("case ", "")),
#     )

#     for case_dir in case_dirs:
#         json_candidates = sorted(case_dir.glob("case_*_results.json"))
#         if not json_candidates:
#             continue

#         json_path = json_candidates[0]
#         case_num = parse_case_num_from_json(json_path)
#         roi_path = roi_root / f"case_{case_num}.png"

#         records.append(
#             {
#                 "case_number": case_num,
#                 "json_path": json_path,
#                 "roi_path": roi_path,
#             }
#         )

#     return records


# def process_single_case(record, output_root: Path):
#     """Process one case and save both JSON + alignment image in one case folder."""
#     case_num = record["case_number"]
#     json_path = record["json_path"]
#     roi_path = record["roi_path"]

#     if not json_path.exists():
#         return False, "JSON not found", None
#     if not roi_path.exists():
#         return False, f"ROI not found: {roi_path.name}", None

#     roi_img = load_roi_image(roi_path)
#     tooth_data = load_tooth_json(json_path)
#     caries_results = generate_caries_mapping(roi_img, tooth_data, case_num)

#     case_output_dir = output_root / f"case {case_num}"
#     case_output_dir.mkdir(parents=True, exist_ok=True)

#     caries_json_path = case_output_dir / f"case_{case_num}_caries_mapping.json"
#     with open(caries_json_path, "w", encoding="utf-8") as f:
#         json.dump(
#             {
#                 "case_number": int(case_num),
#                 "teeth_caries_data": caries_results,
#             },
#             f,
#             indent=2,
#         )

#     combined_img, stats, overlap_stats = create_detailed_alignment_image(roi_img, tooth_data)
#     alignment_img_path = case_output_dir / f"case_{case_num}_alignment_detailed.png"
#     cv2.imwrite(str(alignment_img_path), combined_img)

#     teeth_with_caries = sum(1 for r in caries_results if r["has_caries"])
#     msg = f"{len(caries_results)} teeth, {teeth_with_caries} with caries"

#     case_stats = {
#         "stats": stats,
#         "overlap_stats": overlap_stats,
#         "caries_results": caries_results,
#     }
#     return True, msg, case_stats


# def run_caries_mapping_pipeline(
#     segmentation_root: Path,
#     roi_root: Path,
#     output_root: Path,
#     max_cases=None,
# ):
#     """Run mapping pipeline and keep all outputs under one root folder."""
#     output_root.mkdir(parents=True, exist_ok=True)

#     records = collect_case_records(segmentation_root, roi_root)
#     if not records:
#         raise FileNotFoundError(
#             "No case JSON files found. Check segmentation_root path and Step 1 outputs."
#         )

#     if max_cases is not None:
#         records = records[:max_cases]
#         print(f"Limiting execution to {max_cases} sample cases.")

#     summary_stats = {
#         "total_cases": len(records),
#         "processed_cases": 0,
#         "failed_cases": 0,
#         "total_teeth": 0,
#         "teeth_with_caries": 0,
#     }
#     all_caries_results = []

#     print("=" * 70)
#     print("Dental Caries Mapping Pipeline")
#     print("=" * 70)
#     print(f"Segmentation JSON source: {segmentation_root}")
#     print(f"ROI source: {roi_root}")
#     print(f"Output root: {output_root}")
#     print("=" * 70)

#     pbar = tqdm(records, desc="Processing cases", unit="case")
#     for record in pbar:
#         case_num = record["case_number"]
#         try:
#             success, msg, payload = process_single_case(record, output_root=output_root)

#             if success and payload:
#                 case_results = payload["caries_results"]
#                 summary_stats["processed_cases"] += 1
#                 summary_stats["total_teeth"] += len(case_results)
#                 summary_stats["teeth_with_caries"] += sum(
#                     1 for r in case_results if r["has_caries"]
#                 )

#                 for r in case_results:
#                     csv_result = {k: v for k, v in r.items() if k != "caries_coordinates"}
#                     all_caries_results.append(csv_result)

#                 pbar.set_postfix_str(f"case {case_num}: {msg}")
#             else:
#                 summary_stats["failed_cases"] += 1
#                 pbar.set_postfix_str(f"case {case_num}: FAIL - {msg}")

#         except Exception as e:
#             summary_stats["failed_cases"] += 1
#             pbar.set_postfix_str(f"case {case_num}: ERROR - {str(e)[:40]}")

#     if all_caries_results:
#         df = pd.DataFrame(all_caries_results)
#         csv_path = output_root / "caries_mapping_results.csv"
#         df.to_csv(csv_path, index=False)
#         print(f"\nResults CSV saved to: {csv_path}")

#         summary_df = (
#             df.groupby(["case_number"]).agg(
#                 {
#                     "tooth_id": "count",
#                     "caries_pixels": "sum",
#                     "has_caries": "sum",
#                     "caries_percentage": "mean",
#                 }
#             ).rename(
#                 columns={
#                     "tooth_id": "total_teeth",
#                     "has_caries": "teeth_with_caries",
#                     "caries_percentage": "avg_caries_percentage",
#                 }
#             )
#         )
#         summary_csv_path = output_root / "caries_summary_by_case.csv"
#         summary_df.to_csv(summary_csv_path)
#         print(f"Summary CSV saved to: {summary_csv_path}")

#     print("\n" + "=" * 70)
#     print("Processing Complete")
#     print("=" * 70)
#     print(f"Total cases found: {summary_stats['total_cases']}")
#     print(f"Processed: {summary_stats['processed_cases']}")
#     print(f"Failed: {summary_stats['failed_cases']}")
#     print(f"Total teeth analyzed: {summary_stats['total_teeth']}")
#     print(f"Teeth with caries: {summary_stats['teeth_with_caries']}")
#     if summary_stats["total_teeth"] > 0:
#         pct = summary_stats["teeth_with_caries"] / summary_stats["total_teeth"] * 100
#         print(f"Caries prevalence: {pct:.2f}%")
#     print("=" * 70)

#     return all_caries_results, summary_stats


# # =============================================================================
# # Notebook Runner (Current Project Environment)
# # =============================================================================

# project_root = Path.cwd()

# # Step 1 output folder (refactored pipeline)
# segmentation_root = project_root / "segmentation+recognition-dataset"
# if not segmentation_root.exists():
#     raise FileNotFoundError(
#         f"Segmentation output folder not found: {segmentation_root}. Run Step 1 first."
#     )

# # ROI folder from raw_data
# roi_root = project_root.parent / "data" / "500-roi"
# if not roi_root.exists():
#     raise FileNotFoundError(
#         f"ROI folder not found: {roi_root}."
#     )

# # Single output root as requested
# output_root = project_root / "caries_mapping_output"

# # RUN ON ALL CASES
# all_caries_results, summary_stats = run_caries_mapping_pipeline(
#     segmentation_root=segmentation_root,
#     roi_root=roi_root,
#     output_root=output_root,
#     # max_cases=5
# )

# # Uncomment the below block to run on ALL cases
# # all_caries_results, summary_stats = run_caries_mapping_pipeline(
# #     segmentation_root=segmentation_root,
# #     roi_root=roi_root,
# #     output_root=output_root,
# # )

Limiting execution to 5 sample cases.
Dental Caries Mapping Pipeline
Segmentation JSON source: c:\Users\jaopi\Desktop\SP\phase2-1april\segmentation+recognition-dataset
ROI source: c:\Users\jaopi\Desktop\SP\data\500-roi
Output root: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output


Processing cases: 100%|██████████| 5/5 [00:08<00:00,  1.77s/case, case 5: 23 teeth, 7 with caries]


Results CSV saved to: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output\caries_mapping_results.csv
Summary CSV saved to: c:\Users\jaopi\Desktop\SP\phase2-1april\caries_mapping_output\caries_summary_by_case.csv

Processing Complete
Total cases found: 5
Processed: 5
Failed: 0
Total teeth analyzed: 145
Teeth with caries: 33
Caries prevalence: 22.76%


## 3. PCA-based Tooth Alignment + Surface Classification & Visualization

This module applies PCA-based alignment to each detected tooth and classifies
caries lesions into 3 anatomical surfaces using Diagonal-from-Centroid zone voting:
- **Occlusal** — pixels nearest to the crown (vertical zone)
- **Mesial**   — pixels nearest to the midline side (horizontal zone)
- **Distal**   — pixels nearest to the distal side (horizontal zone)

Supports multiple lesions per tooth (each disconnected cluster classified independently).

Visualization features:
- Aligned tooth polygon with 4-triangle zone overlay (diagonal from centroid)
- Caries pixels coloured by predicted surface
- Gold star (★) = tooth centroid; diagonal guide-lines to all 4 corners
- Per-instance PNG: `tooth_{id}_instance{n}_{surface}.png`

Output Structure:
```
PCA_Output/case_{X}/
├── case_{X}.json           ← per-tooth classification + all instances
└── tooth_{id}/
    ├── tooth_{id}_instance0_Distal.png
    └── tooth_{id}_instance1_Mesial.png
```


In [ ]:
# PCA & Surface Classification [v4.5 + v4.6 in one run]
# v4.5: X-thirds dominant zone
# v4.6: Diagonal-from-Centroid (4-triangle zone split)

import os, json, math
import warnings
from pathlib import Path
from typing import Any, Callable

import pandas as pd
import numpy as np
import cv2
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit

# Relative paths — resolved from the SP root (one level above this notebook)
_SP_DIR    = Path.cwd().parent
SEG_DIR    = str(_SP_DIR / "week2-Tooth Detection & Segmentation" / "500-segmentation+recognition")
CARIES_DIR = str(_SP_DIR / "week3-Caries-to-Tooth Mapping" / "dental_analysis_output")

MAX_TILT_DEG      = 45.0   # clamp extreme PCA angles (bad tooth masks)
MIN_CLUSTER_SIZE  = 15     # noise removal: drop clusters smaller than this
LEFT_BOUND  = 0.40   # X-split: Left zone  0.00-0.40 (wider = fewer M/D->Occ spillover)
RIGHT_BOUND = 0.60   # X-split: Right zone 0.60-1.00 | Center = 0.40-0.60 (Occlusal)

SURFACE_COLORS = {"Occlusal": "#E74C3C", "Mesial": "#3498DB",
                  "Distal": "#27AE60", "Other": "#2ECC71", -1: "#95A5A6"}

def is_upper_jaw(tid): return int(str(tid)[0]) in [1, 2]
def get_quadrant(tid):  return int(str(tid)[0])

def load_seg(case_id):
    path = os.path.join(SEG_DIR, f"case {case_id}", f"case_{case_id}_results.json")
    if not os.path.exists(path): print(f"[ERROR] SEG: {path}"); return None
    with open(path) as f: return json.load(f)

def load_caries(case_id):
    path = os.path.join(CARIES_DIR, f"case {case_id}", f"case_{case_id}_caries_mapping.json")
    if not os.path.exists(path): print(f"[ERROR] CARIES: {path}"); return None
    with open(path) as f: return json.load(f)

def build_seg_map(seg_data):
    return {str(t["tooth_id"]): t.get("pixel_coordinates", []) for t in seg_data.get("teeth_data", [])}

def get_caries_list(d): return d.get("teeth_caries_data", [])
def compute_centroid(p): a=np.array(p,dtype=np.float64); return float(np.mean(a[:,0])),float(np.mean(a[:,1]))

def get_bbox(pts):
    p=np.array(pts,dtype=np.float64); mn,mx=np.min(p,0),np.max(p,0)
    return mn[0],mn[1],mx[0]-mn[0],mx[1]-mn[1]


# =============================================================================
# NOISE REMOVAL (from week7)
# =============================================================================
def remove_small_clusters(caries_pts, min_cluster=MIN_CLUSTER_SIZE):
    if len(caries_pts) < min_cluster:
        return caries_pts
    pts = np.array(caries_pts, dtype=np.int32)
    x_min, y_min = pts.min(axis=0)
    x_max, y_max = pts.max(axis=0)
    pad = 2
    w = x_max - x_min + 1 + 2*pad
    h = y_max - y_min + 1 + 2*pad
    mask = np.zeros((h, w), dtype=np.uint8)
    shifted = pts - np.array([x_min - pad, y_min - pad])
    mask[shifted[:,1], shifted[:,0]] = 255
    n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    keep = np.zeros_like(mask)
    for lbl in range(1, n_labels):
        if stats[lbl, cv2.CC_STAT_AREA] >= min_cluster:
            keep[labels == lbl] = 255
    ys, xs = np.where(keep > 0)
    if len(xs) == 0: return caries_pts
    return np.column_stack([xs + x_min - pad, ys + y_min - pad]).astype(np.float64)


# =============================================================================
# PCA  — 4-Rule Orientation (ported from week7 multi_zone_classifier.py)
# =============================================================================
def perform_pca(points, tooth_id):
    pts = np.array(points, dtype=np.float64).reshape(-1, 2)
    mean = np.mean(pts, axis=0)
    centered = pts - mean

    _, eigvecs = cv2.PCACompute(centered.astype(np.float32), mean=None)
    ev0 = eigvecs[0].astype(np.float64)
    ev1 = eigvecs[1].astype(np.float64)

    if abs(ev0[1]) >= abs(ev1[1]):
        vertical_axis = ev0.copy()
        horizontal_axis = ev1.copy()
    else:
        vertical_axis = ev1.copy()
        horizontal_axis = ev0.copy()

    upper = is_upper_jaw(tooth_id)
    if upper:
        if vertical_axis[1] < 0:
            vertical_axis = -vertical_axis
    else:
        if vertical_axis[1] > 0:
            vertical_axis = -vertical_axis

    quadrant = get_quadrant(tooth_id)
    if quadrant in [1, 4]:
        if horizontal_axis[0] < 0:
            horizontal_axis = -horizontal_axis
    else:
        if horizontal_axis[0] > 0:
            horizontal_axis = -horizontal_axis

    angle_from_x = math.atan2(vertical_axis[1], vertical_axis[0])
    if upper:
        target_angle = math.pi / 2
    else:
        target_angle = -math.pi / 2

    rotation_angle = target_angle - angle_from_x

    while rotation_angle > math.pi:
        rotation_angle -= 2 * math.pi
    while rotation_angle < -math.pi:
        rotation_angle += 2 * math.pi

    clamped = False
    if abs(math.degrees(rotation_angle)) > MAX_TILT_DEG:
        rotation_angle = 0.0
        clamped = True

    return mean, rotation_angle, clamped


def rotate(pts, center, angle):
    p = np.array(pts, dtype=np.float64) - center
    c, s = np.cos(angle), np.sin(angle)
    return np.dot(p, np.array([[c,-s],[s,c]]).T) + center


ClassifierFn = Callable[[str, list, list], tuple[str, float, dict[str, Any]]]


# =============================================================================
# CLASSIFICATION  v4.5 — X-thirds dominant zone (week7 approach)
# =============================================================================
def classify_surface_v45(tid: str, tooth_pts: list, caries_pts: list) -> tuple[str, float, dict[str, Any]]:
    """
    FDI anatomy reference (after 4-rule PCA, horizontal axis enforced):
      Q1/Q4 (image left):  Left third = Distal | Center = Occlusal | Right third = Mesial
      Q2/Q3 (image right): Left third = Mesial | Center = Occlusal | Right third = Distal

    Dominant zone (most pixels) determines the prediction.
    """
    # Step 1: Noise removal
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    # Step 2: PCA alignment (4-rule)
    center, angle, clamped = perform_pca(tooth_pts, tid)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    rel_xs = np.clip((caries_rot[:,0] - x) / w, 0.0, 1.0)
    n_pts  = len(rel_xs)

    # Step 3: X-thirds zone assignment
    # After Rule 3 enforcement, horizontal axis is consistent per quadrant:
    #   Q1/Q4: +X is toward Mesial -> right third = Mesial, left third = Distal
    #   Q2/Q3: -X is toward Mesial -> left third = Mesial, right third = Distal
    quadrant = get_quadrant(tid)
    if quadrant in [1, 4]:
        # Q1/Q4: low x = Distal, center = Occlusal, high x = Mesial
        d_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        m_mask = rel_xs > RIGHT_BOUND
    else:
        # Q2/Q3: low x = Mesial, center = Occlusal, high x = Distal
        m_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        d_mask = rel_xs > RIGHT_BOUND

    m_count = int(np.sum(m_mask))  # Mesial
    c_count = int(np.sum(c_mask))  # Occlusal (center)
    d_count = int(np.sum(d_mask))  # Distal

    # Step 4: Dominant zone wins
    vote_map = {"Mesial": m_count, "Occlusal": c_count, "Distal": d_count}
    winner   = max(vote_map, key=vote_map.get)

    vote_fractions = {k: round(v/max(n_pts,1), 4) for k,v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped
    return winner, float(math.degrees(angle)), vote_fractions


# =============================================================================
# CLASSIFICATION  v4.6 — Diagonal-from-Centroid (4-triangle zone split)
# =============================================================================
def classify_surface_v46(tid: str, tooth_pts: list, caries_pts: list) -> tuple[str, float, dict[str, Any]]:
    """
    After 4-rule PCA rotation:
      - Build tooth bbox
      - Split by diagonals crossing bbox centroid
      - Horizontal triangles -> Mesial/Distal by quadrant
      - Vertical triangles -> Occlusal
    """
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    center, angle, clamped = perform_pca(tooth_pts, tid)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    cx = x + (w / 2.0)
    cy = y + (h / 2.0)
    quadrant = get_quadrant(tid)

    vote_map = {"Mesial": 0, "Occlusal": 0, "Distal": 0}

    for px, py in caries_rot:
        dx = float(px - cx)
        dy = float(py - cy)

        # left/right triangles vs top/bottom triangles
        if abs(dx / w) > abs(dy / h):
            if quadrant in [1, 4]:
                zone = "Mesial" if dx > 0 else "Distal"
            else:
                zone = "Distal" if dx > 0 else "Mesial"
        else:
            zone = "Occlusal"

        vote_map[zone] += 1

    winner = max(vote_map, key=vote_map.get)
    n_pts = len(caries_rot)

    vote_fractions = {k: round(v/max(n_pts,1), 4) for k,v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped
    return winner, float(math.degrees(angle)), vote_fractions

# =============================================================================
# classify_biaxial - เพิ่มแกน Y (Top 30% → Occlusal)
# =============================================================================
def classify_biaxial(tid: str, tooth_pts: list, caries_pts: list) -> tuple[str, float, dict[str, Any]]:
    # 1. กรอง Noise และทำ PCA (เหมือนเดิม)
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    center, angle, clamped = perform_pca(tooth_pts, tid)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    # 2. หาพิกัดสัมพัทธ์ทั้งแกน X และแกน Y
    rel_xs = np.clip((caries_rot[:,0] - x) / w, 0.0, 1.0)
    rel_ys = np.clip((caries_rot[:,1] - y) / h, 0.0, 1.0)
    n_pts  = len(rel_xs)

    quadrant = get_quadrant(tid)
    
    # 3. หั่นแบ่งแกน X (X-Thirds เหมือนเดิม)
    if quadrant in [1, 4]:
        d_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        m_mask = rel_xs > RIGHT_BOUND
    else:
        m_mask = rel_xs < LEFT_BOUND
        c_mask = (rel_xs >= LEFT_BOUND) & (rel_xs <= RIGHT_BOUND)
        d_mask = rel_xs > RIGHT_BOUND

    m_count = int(np.sum(m_mask))
    c_count = int(np.sum(c_mask))
    d_count = int(np.sum(d_mask))

    # 4. เพิ่มลอจิกแกน Y (Crown Zone: 30% ของความสูงฟัน)
    is_upper = quadrant in [1, 2]
    if is_upper:
        # ฟันบน ยอดฟันอยู่ด้านล่างของภาพ (Y มีค่ามาก)
        crown_mask = rel_ys >= 0.70
    else:
        # ฟันล่าง ยอดฟันอยู่ด้านบนของภาพ (Y มีค่าน้อย)
        crown_mask = rel_ys <= 0.30

    crown_count = int(np.sum(crown_mask))

    # 5. Biaxial Override: ถ้ารอยผุส่วนใหญ่อยู่บนยอดฟัน (มากกว่า 50%)
    if crown_count > (n_pts * 0.5):
        c_count += crown_count  # เทคะแนนบวกเพิ่มให้ Occlusal อย่างมีนัยสำคัญ

    vote_map = {"Mesial": m_count, "Occlusal": c_count, "Distal": d_count}
    winner   = max(vote_map, key=vote_map.get)

    vote_fractions = {k: round(v/max(n_pts,1), 4) for k,v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped
    vote_fractions["crown_ratio"] = round(crown_count / max(n_pts, 1), 4)
    return winner, float(math.degrees(angle)), vote_fractions

# =============================================================================
# cllassify_fuzzy
# =============================================================================
def classify_fuzzy(tid: str, tooth_pts: list, caries_pts: list) -> tuple[str, float, dict[str, Any]]:
    # 1. กรอง Noise และทำ PCA
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return "Other", 0.0, {}

    center, angle, clamped = perform_pca(tooth_pts, tid)
    tooth_rot   = rotate(tooth_pts,    center, angle)
    caries_rot  = rotate(caries_clean, center, angle)

    x, y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return "Other", float(math.degrees(angle)), {}

    # 2. หาพิกัดสัมพัทธ์แกน X และ Y (0.0 ถึง 1.0)
    rel_xs = np.clip((caries_rot[:,0] - x) / w, 0.0, 1.0)
    rel_ys = np.clip((caries_rot[:,1] - y) / h, 0.0, 1.0)
    n_pts  = len(rel_xs)

    quadrant = get_quadrant(tid)
    is_upper = quadrant in [1, 2]

    # 3. กำหนดพิกัด Focal Points สมมติ (X, Y)
    # ฟันบนยอดอยู่ล่าง (Y=0.8), ฟันล่างยอดอยู่บน (Y=0.2)
    crown_y = 0.85 if is_upper else 0.15 
    
    focal_center = (0.5, crown_y) # จุด Occlusal
    focal_left   = (0.0, 0.5)     # จุดขอบซ้าย
    focal_right  = (1.0, 0.5)     # จุดขอบขวา

    vote_left = 0.0
    vote_center = 0.0
    vote_right = 0.0

    # 4. Fuzzy Voting: คำนวณระยะห่างแบบ Euclidean แล้วแปลงเป็นคะแนนน้ำหนัก
    for rx, ry in zip(rel_xs, rel_ys):
        # ระยะห่างกำลังสอง (Squared Distance) เพื่อประสิทธิภาพ
        d2_center = (rx - focal_center[0])**2 + (ry - focal_center[1])**2
        d2_left   = (rx - focal_left[0])**2   + (ry - focal_left[1])**2
        d2_right  = (rx - focal_right[0])**2  + (ry - focal_right[1])**2

        # น้ำหนักแปรผกผันกับระยะทาง (บวก epsilon 0.01 กันหารด้วย 0)
        # ยิ่งระยะห่างน้อย (d2 ต่ำ) weight ยิ่งมีค่าสูง
        vote_center += 1.0 / (d2_center + 0.05)
        vote_left   += 1.0 / (d2_left + 0.05)
        vote_right  += 1.0 / (d2_right + 0.05)

    # 5. แมปผลลัพธ์ Left/Right กลับเป็น Mesial/Distal ตาม Quadrant
    if quadrant in [1, 4]:
        vote_map = {"Distal": vote_left, "Occlusal": vote_center, "Mesial": vote_right}
    else:
        vote_map = {"Mesial": vote_left, "Occlusal": vote_center, "Distal": vote_right}

    winner = max(vote_map, key=vote_map.get)

    # แปลงคะแนนดิบเป็นเปอร์เซ็นต์สำหรับเก็บ Log
    total_votes = sum(vote_map.values())
    vote_fractions = {k: round(v/total_votes, 4) for k, v in vote_map.items()}
    vote_fractions["pca_clamped"] = clamped
    
    return winner, float(math.degrees(angle)), vote_fractions

# =============================================================================
# Run 3 (classify_ml)
# =============================================================================
FEATURE_COLS = [
    'is_upper', 'x_mean', 'y_mean', 'x_std', 'y_std',
    'x_min', 'x_max', 'y_min', 'y_max', 'x_range', 'y_range',
    'x_centroid_dist', 'aspect_ratio', 'coverage',
]

rf_model = None
RUN3_MODEL_PATH = Path.cwd() / 'rf_classify_ml.pkl'
RUN3_OUTPUT_ROOT = Path.cwd() / 'PCA_Output_Run3'
RUN3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

def _extract_ml_feature_dict(tid, tooth_pts, caries_pts):
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return None

    center, angle, _ = perform_pca(tooth_pts, tid)
    tooth_rot = rotate(tooth_pts, center, angle)
    caries_rot = rotate(caries_clean, center, angle)

    xy_min = np.min(tooth_rot, axis=0)
    xy_range = np.ptp(tooth_rot, axis=0)
    if xy_range[0] <= 0 or xy_range[1] <= 0:
        return None

    x_rel = np.clip((caries_rot[:, 0] - xy_min[0]) / xy_range[0], 0.0, 1.0)
    y_rel = np.clip((caries_rot[:, 1] - xy_min[1]) / xy_range[1], 0.0, 1.0)

    return {
        'is_upper': 1 if int(str(tid)[0]) in [1, 2] else 0,
        'x_mean': float(np.mean(x_rel)),
        'y_mean': float(np.mean(y_rel)),
        'x_std': float(np.std(x_rel)),
        'y_std': float(np.std(y_rel)),
        'x_min': float(np.min(x_rel)),
        'x_max': float(np.max(x_rel)),
        'y_min': float(np.min(y_rel)),
        'y_max': float(np.max(y_rel)),
        'x_range': float(np.max(x_rel) - np.min(x_rel)),
        'y_range': float(np.max(y_rel) - np.min(y_rel)),
        'x_centroid_dist': float(abs(np.mean(x_rel) - 0.5)),
        'aspect_ratio': float(xy_range[0] / xy_range[1]),
        'coverage': float(len(caries_clean) / (len(tooth_pts) + 1e-6)),
    }

def _ensure_rf_model():
    global rf_model
    if rf_model is not None:
        return rf_model
    if RUN3_MODEL_PATH.exists():
        rf_model = joblib.load(RUN3_MODEL_PATH)
    return rf_model

def classify_ml(tid, tooth_pts, caries_pts):
    model = _ensure_rf_model()
    if model is None:
        return classify_surface_v45(tid, tooth_pts, caries_pts)

    features = _extract_ml_feature_dict(tid, tooth_pts, caries_pts)
    if features is None:
        return classify_surface_v45(tid, tooth_pts, caries_pts)

    feature_df = pd.DataFrame([[features[col] for col in FEATURE_COLS]], columns=FEATURE_COLS)
    try:
        with warnings.catch_warnings():
            warnings.simplefilter('ignore')
            proba = model.predict_proba(feature_df)[0]
        classes = list(model.classes_)
        valid = ['Occlusal', 'Mesial', 'Distal']
        scores = {cls: proba[classes.index(cls)] for cls in valid if cls in classes}
        if not scores:
            return classify_surface_v45(tid, tooth_pts, caries_pts)
        winner = max(scores, key=scores.get)
        return winner, 0.0, {'method': 'RandomForest_Proba'}
    except Exception:
        return classify_surface_v45(tid, tooth_pts, caries_pts)

def process_case_ml(case_id: int, output_root: str | Path = RUN3_OUTPUT_ROOT):
    seg_data = load_seg(case_id)
    caries_data = load_caries(case_id)
    if seg_data is None or caries_data is None:
        return False, 'missing input files'

    out_dir = Path(output_root) / f'case_{case_id}'
    out_dir.mkdir(parents=True, exist_ok=True)

    seg_map = build_seg_map(seg_data)
    results = { 'case_number': case_id, 'teeth_data': [] }
    for tooth in get_caries_list(caries_data):
        tid = str(tooth.get('tooth_id'))
        tooth_pts = seg_map.get(tid, [])
        caries_pts = tooth.get('caries_coordinates', [])
        surface, angle, vote_fractions = classify_ml(tid, tooth_pts, caries_pts)
        results['teeth_data'].append({
            'tooth_id': tid,
            'predicted_surface_fine': surface,
            'caries_position_detail': surface,
            'rotation_angle_deg': angle,
            'vote_fractions': vote_fractions,
            'tooth_coordinates': tooth_pts,
            'caries_coordinates': caries_pts,
        })

    with open(out_dir / f'case_{case_id}.json', 'w') as f:
        json.dump(results, f, indent=2)
    return True, f"saved {case_id}"

# =============================================================================
# end of classification methods
# =============================================================================


def compute_caries_stats(caries_pts, tooth_pts):
    cp, tp = len(caries_pts), len(tooth_pts)
    return cp, (cp/tp*100) if tp else 0


def visualize_tooth_with_zones(
    tooth_pts,
    caries_pts,
    tooth_id,
    classification,
    version: str,
    vote_fractions: dict[str, Any] | None = None,
    save_path: str | Path | None = None,
) -> plt.Figure:
    tooth_pts  = np.array(tooth_pts,  dtype=np.float64)
    caries_pts = np.array(caries_pts, dtype=np.float64)
    x, y = np.min(tooth_pts, axis=0); w, h = np.ptp(tooth_pts, axis=0)

    fig, ax = plt.subplots(figsize=(6, 6))
    q = get_quadrant(tooth_id)

    if version == "v4.5":
        # Q1/Q4: left=Distal, center=Occlusal, right=Mesial
        # Q2/Q3: left=Mesial, center=Occlusal, right=Distal
        left_col  = SURFACE_COLORS["Distal"]  if q in [1,4] else SURFACE_COLORS["Mesial"]
        right_col = SURFACE_COLORS["Mesial"] if q in [1,4] else SURFACE_COLORS["Distal"]
        t1 = x + w*LEFT_BOUND; t2 = x + w*RIGHT_BOUND
        ax.fill([x,t1,t1,x],          [y,y,y+h,y+h], alpha=0.18, color=left_col)
        ax.fill([t1,t2,t2,t1],         [y,y,y+h,y+h], alpha=0.18, color=SURFACE_COLORS["Occlusal"])
        ax.fill([t2,x+w,x+w,t2],       [y,y,y+h,y+h], alpha=0.18, color=right_col)
    else:
        # v4.6: draw diagonal boundaries through tooth centroid
        ax.plot([x, x + w], [y, y + h], linestyle="--", linewidth=1.2, color="gold", alpha=0.6)
        ax.plot([x, x + w], [y + h, y], linestyle="--", linewidth=1.2, color="gold", alpha=0.6)

    ax.scatter(tooth_pts[:,0], tooth_pts[:,1], c="gray", s=2, alpha=0.4)
    color = SURFACE_COLORS.get(classification, "#95A5A6")
    if len(caries_pts)>0:
        ax.scatter(caries_pts[:,0],caries_pts[:,1],c=color,s=10,zorder=5)

    cx,cy = compute_centroid(caries_pts)
    ax.plot(cx,cy,"*",markersize=14,color="gold",zorder=6)
    ax.add_patch(plt.Rectangle((x,y),w,h,fill=False,ls="--",lw=1))

    title = f"Tooth {tooth_id} Q{q} — {classification} ({version})"
    if vote_fractions:
        v = vote_fractions
        title += f"\nOcc={v.get('Occlusal',0):.2f} M={v.get('Mesial',0):.2f} D={v.get('Distal',0):.2f}"

    ax.set_title(title,fontsize=9)
    ax.set_aspect("equal")
    ax.invert_yaxis()
    ax.axis("off")

    if save_path:
        Path(save_path).parent.mkdir(parents=True, exist_ok=True)
        fig.savefig(save_path,dpi=150,bbox_inches="tight")
        plt.close(fig)

    return fig


def process_case(case_id: int, out_dir: str, classifier: ClassifierFn, version: str) -> None:
    seg_data    = load_seg(case_id)
    caries_data = load_caries(case_id)
    if seg_data is None or caries_data is None:
        return

    seg_map     = build_seg_map(seg_data)
    caries_list = get_caries_list(caries_data)
    results = {"case_number": case_id, "teeth_data": []}

    for tooth in caries_list:
        tid        = str(tooth["tooth_id"])
        caries_pts = tooth.get("caries_coordinates", [])
        tooth_pts  = seg_map.get(tid, [])
        if len(caries_pts)==0 or len(tooth_pts)<10:
            continue

        surface, angle, vf = classifier(tid, tooth_pts, caries_pts)
        pixels, pct = compute_caries_stats(caries_pts, tooth_pts)
        results["teeth_data"].append({
            "tooth_id": tid,
            "version": version,
            "has_caries": True,
            "confidence": tooth.get("confidence",0),
            "caries_position_detail": surface,
            "predicted_surface_fine": surface,
            "vote_fractions": vf,
            "rotation_angle": round(angle,2),
            "tooth_coordinates": tooth_pts,
            "caries_coordinates": caries_pts,
            "caries_pixels": pixels,
            "caries_percentage": round(pct,4)
        })

    case_dir = os.path.join(out_dir, f"case_{case_id}")
    os.makedirs(case_dir, exist_ok=True)
    with open(os.path.join(case_dir, f"case_{case_id}.json"), "w") as f:
        json.dump(results, f, indent=4)


def process_case_visual(case_id: int, out_dir: str, classifier: ClassifierFn, version: str) -> None:
    _ = classifier  # kept in signature by design for versioned pipeline symmetry

    case_dir = Path(out_dir) / f"case_{case_id}"
    jp = case_dir / f"case_{case_id}.json"
    if not jp.exists():
        return

    with open(jp) as f:
        data = json.load(f)

    for tooth in data["teeth_data"]:
        tid = tooth["tooth_id"]
        tp = tooth["tooth_coordinates"]
        cp = tooth["caries_coordinates"]

        c, a, _ = perform_pca(tp, tid)
        tr = rotate(tp, c, a)
        cr = rotate(np.array(cp, dtype=np.float64), c, a)

        td = case_dir / f"tooth_{tid}"
        td.mkdir(parents=True, exist_ok=True)

        visualize_tooth_with_zones(
            tr,
            cr,
            tid,
            tooth["caries_position_detail"],
            version=version,
            vote_fractions=tooth.get("vote_fractions",{}),
            save_path=td / f"tooth_{tid}_visual.png",
        )


# VERSION_CONFIGS = {
#     "v4.5": {
#         "out_dir": "PCA_Output_v4.5",
#         "classifier": classify_surface_v45,
#         "label": "X-thirds dominant zone",
#     },
#     "v4.6": {
#         "out_dir": "PCA_Output_v4.6",
#         "classifier": classify_surface_v46,
#         "label": "Diagonal-from-Centroid (4-triangle)",
#     },
# }

VERSION_CONFIGS = {
    "Baseline": {
        "out_dir": "PCA_Output_Baseline",
        "classifier": classify_surface_v45, # ฟังก์ชันเดิม (เปลี่ยนชื่อเป็น classify_xthird ก็ได้)
        "label": "X-Thirds Hard Partition (Baseline)",
    },
    # "Run1": {
    #     "out_dir": "PCA_Output_Run1",
    #     "classifier": classify_biaxial,
    #     "label": "Biaxial Threshold Classification",
    # },
    # "Run2": {
    #     "out_dir": "PCA_Output_Run2",
    #     "classifier": classify_fuzzy,
    #     "label": "Fuzzy Distance-Based Classification",
    # },
    "Run3": {
        "out_dir": "PCA_Output_Run3",
        "classifier": classify_ml,
        "label": "Geometric Feature + ML Classifier",
    }
}


for version, cfg in VERSION_CONFIGS.items():
    print(f"\n{'='*60}")
    print(f"  Running {version} — {cfg['label']}")
    print(f"  Output → {cfg['out_dir']}/")
    print(f"{'='*60}")

    os.makedirs(cfg["out_dir"], exist_ok=True)

    for cid in range(1, 501):
        if cid % 50 == 1:
            print(f"  [{version}] case {cid}...")
        process_case(cid, cfg["out_dir"], cfg["classifier"], version)
        process_case_visual(cid, cfg["out_dir"], cfg["classifier"], version)

print("\n[ALL DONE] Both versions complete.")


  Running Baseline — X-Thirds Hard Partition (Baseline)
  Output → PCA_Output_Baseline/
  [Baseline] case 1...
  [Baseline] case 51...
  [Baseline] case 101...
  [Baseline] case 151...
  [Baseline] case 201...
  [Baseline] case 251...
  [Baseline] case 301...
  [Baseline] case 351...
  [Baseline] case 401...
  [Baseline] case 451...

  Running Run3 — Geometric Feature + ML Classifier
  Output → PCA_Output_Run3/
  [Run3] case 1...
  [Run3] case 51...
  [Run3] case 101...
  [Run3] case 151...
  [Run3] case 201...
  [Run3] case 251...
  [Run3] case 301...
  [Run3] case 351...
  [Run3] case 401...
  [Run3] case 451...

[ALL DONE] Both versions complete.


# RUN 3: Random Forest classify_ml (cleaned, SP-root paths)
ได้ rf_model ที่ขั้นตอนนี้

In [8]:
# =========================================================
# RUN 3: Random Forest classify_ml (cleaned, SP-root paths)
# =========================================================
import warnings
import joblib
import json
import pandas as pd
import numpy as np
import math
import cv2
from pathlib import Path
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
 )

FEATURE_COLS = [
    'is_upper', 'x_mean', 'y_mean', 'x_std', 'y_std',
    'x_min', 'x_max', 'y_min', 'x_range', 'y_range',
    'x_centroid_dist', 'aspect_ratio', 'coverage',
 ]

# model placeholders
rf_model = None
rf_feature_cols = FEATURE_COLS

# ==========================================
# 1. การตั้งค่า Path (resolved from SP root)
# ==========================================
# Relative paths — resolved from the SP root (one level above this notebook)
_SP_DIR    = Path.cwd().parent
SEG_DIR    = str(_SP_DIR / "week2-Tooth Detection & Segmentation" / "500-segmentation+recognition")
CARIES_DIR = str(_SP_DIR / "week3-Caries-to-Tooth Mapping" / "dental_analysis_output")

# Keep Path objects used elsewhere for backwards compatibility
RUN3_SEG_ROOT = Path(SEG_DIR)
RUN3_CARIES_ROOT = Path(CARIES_DIR)
RUN3_GT_ROOT = _SP_DIR / "data" / "500 cases with annotation"

RUN3_OUTPUT_ROOT = Path.cwd() / 'PCA_Output_Run3'
RUN3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

VALID_SURFACES = ["Occlusal", "Mesial", "Distal", "Other"]

# Fallback ground-truth parser (when evaluation cell has not been run)
if "parse_case_ground_truth" not in globals():
    try:
        import sys
        _WEEK6_DIR = _SP_DIR / "week6"
        if str(_WEEK6_DIR) not in sys.path:
            sys.path.append(str(_WEEK6_DIR))
        import xml_ground_truth_parser as _gt_parser

        def parse_case_ground_truth(case_folder):
            gt = []
            for xml_file in sorted(Path(case_folder).glob("*.xml")):
                parsed = _gt_parser.parse_aim_xml(str(xml_file))
                if parsed is None:
                    continue
                tooth = str(parsed.get("tooth_fdi", parsed.get("tooth", "Unknown")))
                surface = parsed.get("surface_name", parsed.get("surface", "Other"))
                if surface not in VALID_SURFACES:
                    surface = "Other"
                gt.append({"tooth": tooth, "surface": surface})
            return gt
    except Exception as e:
        raise RuntimeError(
            "parse_case_ground_truth is missing and fallback import failed. "
            "Run the evaluation cell or check week6/xml_ground_truth_parser.py."
        ) from e

# Fallback helpers (when PCA cell has not been run)
if "is_upper_jaw" not in globals():
    def is_upper_jaw(tid):
        return int(str(tid)[0]) in [1, 2]

if "get_quadrant" not in globals():
    def get_quadrant(tid):
        return int(str(tid)[0])

if "get_bbox" not in globals():
    def get_bbox(pts):
        p = np.array(pts, dtype=np.float64)
        mn, mx = np.min(p, axis=0), np.max(p, axis=0)
        return mn[0], mn[1], mx[0] - mn[0], mx[1] - mn[1]

if "rotate" not in globals():
    def rotate(pts, center, angle):
        p = np.array(pts, dtype=np.float64) - center
        c, s = np.cos(angle), np.sin(angle)
        return np.dot(p, np.array([[c, -s], [s, c]]).T) + center

if "remove_small_clusters" not in globals():
    MIN_CLUSTER_SIZE = 15
    def remove_small_clusters(caries_pts, min_cluster=MIN_CLUSTER_SIZE):
        if len(caries_pts) < min_cluster:
            return caries_pts
        pts = np.array(caries_pts, dtype=np.int32)
        x_min, y_min = pts.min(axis=0)
        x_max, y_max = pts.max(axis=0)
        pad = 2
        w = x_max - x_min + 1 + 2 * pad
        h = y_max - y_min + 1 + 2 * pad
        mask = np.zeros((h, w), dtype=np.uint8)
        shifted = pts - np.array([x_min - pad, y_min - pad])
        mask[shifted[:, 1], shifted[:, 0]] = 255
        n_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
        keep = np.zeros_like(mask)
        for lbl in range(1, n_labels):
            if stats[lbl, cv2.CC_STAT_AREA] >= min_cluster:
                keep[labels == lbl] = 255
        ys, xs = np.where(keep > 0)
        if len(xs) == 0:
            return caries_pts
        return np.column_stack([xs + x_min - pad, ys + y_min - pad]).astype(np.float64)

if "perform_pca" not in globals():
    MAX_TILT_DEG = 45.0
    def perform_pca(points, tooth_id):
        pts = np.array(points, dtype=np.float64).reshape(-1, 2)
        mean = np.mean(pts, axis=0)
        centered = pts - mean

        _, eigvecs = cv2.PCACompute(centered.astype(np.float32), mean=None)
        ev0 = eigvecs[0].astype(np.float64)
        ev1 = eigvecs[1].astype(np.float64)

        if abs(ev0[1]) >= abs(ev1[1]):
            vertical_axis = ev0.copy()
            horizontal_axis = ev1.copy()
        else:
            vertical_axis = ev1.copy()
            horizontal_axis = ev0.copy()

        upper = is_upper_jaw(tooth_id)
        if upper:
            if vertical_axis[1] < 0:
                vertical_axis = -vertical_axis
        else:
            if vertical_axis[1] > 0:
                vertical_axis = -vertical_axis

        quadrant = get_quadrant(tooth_id)
        if quadrant in [1, 4]:
            if horizontal_axis[0] < 0:
                horizontal_axis = -horizontal_axis
        else:
            if horizontal_axis[0] > 0:
                horizontal_axis = -horizontal_axis

        angle_from_x = math.atan2(vertical_axis[1], vertical_axis[0])
        target_angle = math.pi / 2 if upper else -math.pi / 2
        rotation_angle = target_angle - angle_from_x

        while rotation_angle > math.pi:
            rotation_angle -= 2 * math.pi
        while rotation_angle < -math.pi:
            rotation_angle += 2 * math.pi

        clamped = False
        if abs(math.degrees(rotation_angle)) > MAX_TILT_DEG:
            rotation_angle = 0.0
            clamped = True

        return mean, rotation_angle, clamped

if "build_seg_map" not in globals():
    def build_seg_map(seg_data):
        return {
            str(t["tooth_id"]): t.get("pixel_coordinates", [])
            for t in seg_data.get("teeth_data", [])
        }

# Fallback evaluation helpers (when evaluation cell has not been run)
if "load_prediction" not in globals():
    def load_prediction(case_num, out_dir):
        pred_path = Path(out_dir) / f"case_{case_num}" / f"case_{case_num}.json"
        if not pred_path.exists():
            return []
        with open(pred_path, "r") as f:
            data = json.load(f)
        preds = []
        for t in data.get("teeth_data", []):
            tooth = str(t.get("tooth_id", "Unknown"))
            surface = t.get("predicted_surface_fine", t.get("caries_position_detail", "Other"))
            if surface not in VALID_SURFACES:
                surface = "Other"
            preds.append({"tooth": tooth, "surface": surface})
        return preds

if "match_case" not in globals():
    def match_case(gt, pred):
        pred_dict = {p["tooth"]: p["surface"] for p in pred}
        y_true = []
        y_pred = []
        for g in gt:
            tooth = g["tooth"]
            gt_surface = g["surface"]
            pred_surface = pred_dict.get(tooth, "Other")
            y_true.append(gt_surface)
            y_pred.append(pred_surface)
        return y_true, y_pred

if "evaluate_version" not in globals():
    def evaluate_version(version):
        out_dir = f"PCA_Output_{version}"
        base_gt = RUN3_GT_ROOT
        all_y_true = []
        all_y_pred = []
        for case_num in range(1, 501):
            gt_folder = base_gt / f"case {case_num}"
            gt = parse_case_ground_truth(gt_folder)
            pred = load_prediction(case_num, out_dir)
            if len(gt) == 0 and len(pred) == 0:
                continue
            yt, yp = match_case(gt, pred)
            all_y_true.extend(yt)
            all_y_pred.extend(yp)
        accuracy = accuracy_score(all_y_true, all_y_pred)
        precision = precision_score(all_y_true, all_y_pred, average="macro", zero_division=0)
        recall = recall_score(all_y_true, all_y_pred, average="macro", zero_division=0)
        f1 = f1_score(all_y_true, all_y_pred, average="macro", zero_division=0)
        cm = confusion_matrix(all_y_true, all_y_pred, labels=VALID_SURFACES)
        cm_df = pd.DataFrame(cm, index=VALID_SURFACES, columns=VALID_SURFACES)
        print(f"\n========== FINAL EVALUATION [{version}] ==========")
        print(f"Total Samples : {len(all_y_true)}")
        print(f"Accuracy      : {accuracy:.4f}")
        print(f"Precision     : {precision:.4f}")
        print(f"Recall        : {recall:.4f}")
        print(f"F1 Score      : {f1:.4f}")
        print("\nConfusion Matrix:")
        print(cm_df)
        print("\nClassification Report:")
        print(classification_report(
            all_y_true,
            all_y_pred,
            labels=VALID_SURFACES,
            zero_division=0
        ))
        return all_y_true, all_y_pred, f1

# ==========================================
# 2. ฟังก์ชันโหลดไฟล์และสกัด Features
# ==========================================
def _load_json_file(path):
    if not path.exists():
        return None
    with open(path, 'r', encoding='utf-8') as f:
        return json.load(f)

def _load_run3_seg_case(case_id):
    path = RUN3_SEG_ROOT / f'case {case_id}' / f'case_{case_id}_results.json'
    return _load_json_file(path)

def _load_run3_caries_case(case_id):
    path = RUN3_CARIES_ROOT / f'case {case_id}' / f'case_{case_id}_caries_mapping.json'
    return _load_json_file(path)

def _extract_ml_feature_dict(tid, tooth_pts, caries_pts):
    caries_clean = remove_small_clusters(caries_pts)
    if len(caries_clean) == 0:
        return None

    center, angle, _ = perform_pca(tooth_pts, tid)
    tooth_rot = rotate(tooth_pts, center, angle)
    caries_rot = rotate(caries_clean, center, angle)

    bbox_x, bbox_y, w, h = get_bbox(tooth_rot)
    if w <= 0 or h <= 0:
        return None

    x_rel = np.clip((caries_rot[:, 0] - bbox_x) / w, 0.0, 1.0)
    y_rel = np.clip((caries_rot[:, 1] - bbox_y) / h, 0.0, 1.0)

    return {
        'is_upper': 1 if int(str(tid)[0]) in [1, 2] else 0,
        'x_mean': float(np.mean(x_rel)),
        'y_mean': float(np.mean(y_rel)),
        'x_std': float(np.std(x_rel)),
        'y_std': float(np.std(y_rel)),
        'x_min': float(np.min(x_rel)),
        'x_max': float(np.max(x_rel)),
        'y_min': float(np.min(y_rel)),
        'x_range': float(np.max(x_rel) - np.min(x_rel)),
        'y_range': float(np.max(y_rel) - np.min(y_rel)),
        'x_centroid_dist': float(abs(np.mean(x_rel) - 0.5)),
        'aspect_ratio': float(w / h),
        'coverage': float(len(caries_clean) / (len(tooth_pts) + 1e-6)),
    }

def create_ml_dataset(case_ids):
    rows = []
    total_cases = len(case_ids)
    print(f"⚙️ Step 1: เริ่มสกัด Features จากข้อมูล {total_cases} เคส (ขั้นตอนนี้อาจใช้เวลา)...", flush=True)

    for i, case_id in enumerate(case_ids):
        if (i + 1) % 50 == 0 or (i + 1) == total_cases:
            print(f"   -> ดึงข้อมูลถึงเคสที่ {i+1}/{total_cases}...", flush=True)

        seg_data = _load_run3_seg_case(case_id)
        caries_data = _load_run3_caries_case(case_id)
        gt_folder = RUN3_GT_ROOT / f'case {case_id}'

        if seg_data is None or caries_data is None or not gt_folder.exists():
            continue

        gt_list = parse_case_ground_truth(gt_folder)
        gt_dict = {str(item['tooth']): item['surface'] for item in gt_list}
        if not gt_dict:
            continue

        seg_map = build_seg_map(seg_data)
        for tooth in caries_data.get('teeth_caries_data', []):
            tid = str(tooth.get('tooth_id', ''))
            if tid not in gt_dict:
                continue

            tooth_pts = seg_map.get(tid, [])
            caries_pts = tooth.get('caries_coordinates', [])
            if len(caries_pts) == 0 or len(tooth_pts) < 10:
                continue

            features = _extract_ml_feature_dict(tid, tooth_pts, caries_pts)
            if features is None:
                continue

            row = {
                'case_id': int(case_id),
                'tooth_id': tid,
                **features,
                'label': gt_dict[tid],
            }
            rows.append(row)

    columns = ['case_id', 'tooth_id', *FEATURE_COLS, 'label']
    df = pd.DataFrame(rows, columns=columns)
    if not df.empty:
        df = df[columns]
    return df

# ==========================================
# 3. โมเดล ML และการทำนาย
# ==========================================
def train_classify_ml(df):
    if df.empty:
        raise ValueError('ML dataset is empty.')

    gss = GroupShuffleSplit(test_size=0.2, random_state=42)
    train_idx, test_idx = next(gss.split(df, groups=df['case_id']))
    df_train = df.iloc[train_idx].reset_index(drop=True)
    df_test = df.iloc[test_idx].reset_index(drop=True)

    model = RandomForestClassifier(
        class_weight='balanced',
        n_estimators=200,
        random_state=42,
    )
    model.fit(df_train[FEATURE_COLS], df_train['label'])

    global rf_model, rf_feature_cols
    rf_model = model
    rf_feature_cols = FEATURE_COLS

    joblib.dump(rf_model, 'rf_classify_ml.pkl')
    print('Saved model to rf_classify_ml.pkl')
    return model, df_test, FEATURE_COLS

def classify_ml(tid, tooth_pts, caries_pts):
    try:
        if rf_model is None:
            return 'Other', 0.0, {}

        features = _extract_ml_feature_dict(tid, tooth_pts, caries_pts)
        if features is None:
            return 'Other', 0.0, {}

        # Build a single-row DataFrame with explicit column names to match training schema
        feature_values = [features[col] for col in FEATURE_COLS]
        df_features = pd.DataFrame([feature_values], columns=FEATURE_COLS)
        prediction = rf_model.predict(df_features)[0]

        return prediction, 0.0, {"method": "RandomForest"}
    except Exception as e:
        print(e)
        return 'Other', 0.0, {}

def process_case_ml(case_id, output_root):
    seg_data = _load_run3_seg_case(case_id)
    caries_data = _load_run3_caries_case(case_id)

    case_dir = output_root / f'case_{case_id}'
    case_dir.mkdir(parents=True, exist_ok=True)

    result = {'case_number': int(case_id), 'teeth_data': []}

    if seg_data is None or caries_data is None:
        with open(case_dir / f'case_{case_id}.json', 'w', encoding='utf-8') as f:
            json.dump(result, f, indent=2)
        return False, 'Missing input data'

    seg_map = build_seg_map(seg_data)
    caries_list = caries_data.get('teeth_caries_data', [])

    for tooth in caries_list:
        tid = str(tooth.get('tooth_id', ''))
        tooth_pts = seg_map.get(tid, [])
        caries_pts = tooth.get('caries_coordinates', [])

        surface, angle, metadata = classify_ml(tid, tooth_pts, caries_pts)

        result['teeth_data'].append({
            'tooth_id': tid,
            'version': 'Run3',
            'has_caries': True,
            'confidence': float(tooth.get('confidence', 0.0)),
            'caries_position_detail': surface,
            'predicted_surface_fine': surface,
            'tooth_coordinates': tooth_pts,
            'caries_coordinates': caries_pts,
        })

    with open(case_dir / f'case_{case_id}.json', 'w', encoding='utf-8') as f:
        json.dump(result, f, indent=2)

    return True, f"OK ({len(result['teeth_data'])} teeth)"

def run_classify_ml_pipeline(case_ids):
    print("🚀 เริ่มรัน Pipeline Run 3...", flush=True)
    ml_df = create_ml_dataset(case_ids)
    print(f'✅ Step 1 เสร็จสิ้น! ได้ข้อมูลเตรียมเทรนทั้งหมด: {len(ml_df)} ซี่', flush=True)

    print("⚙️ Step 2: กำลัง Train โมเดล Random Forest...", flush=True)
    model, df_test, feature_cols = train_classify_ml(ml_df)
    print(f'✅ Step 2 เสร็จสิ้น! Train: {len(ml_df) - len(df_test)} ซี่ | Test: {len(df_test)} ซี่', flush=True)

    processed = 0
    failed = 0
    total_cases = len(case_ids)

    print("⚙️ Step 3: นำโมเดลไปทำนายผลทั้ง 500 เคส...", flush=True)
    for i, case_id in enumerate(case_ids):
        ok, _ = process_case_ml(case_id, RUN3_OUTPUT_ROOT)
        if ok:
            processed += 1
        else:
            failed += 1

        current_step = i + 1
        if current_step % max(1, total_cases // 10) == 0 or current_step == total_cases:
            percent = (current_step / total_cases) * 100
            print(f"   -> ทำนายผล... {percent:.0f}% ({current_step}/{total_cases} เคส)", flush=True)

    print(f'🎉 สำเร็จ! เขียนไฟล์ทำนายผลแล้ว: {processed} เคส, ล้มเหลว: {failed} เคส', flush=True)
    return model, df_test

# ==========================================
# 4. ลำดับการเรียกใช้งาน (Execution Block)
# ==========================================
# 4.1 ลงทะเบียนเข้า Config
if "VERSION_CONFIGS" not in globals() or VERSION_CONFIGS is None:
    VERSION_CONFIGS = {}
if "Run3" not in VERSION_CONFIGS:
    VERSION_CONFIGS["Run3"] = {
        "out_dir": "PCA_Output_Run3",
        "classifier": classify_ml,
        "label": "Geometric Feature + ML Classifier",
    }

# 4.2 รันสกัดข้อมูล เทรนโมเดล และทำนายผล 500 เคส
run3_case_ids = list(range(1, 501))
rf_model, df_test = run_classify_ml_pipeline(run3_case_ids)

# 4.3 ตรวจคำตอบและโชว์คะแนน (รันหลังจากทำนายเสร็จแล้วเท่านั้น)
all_y_true_Run3, all_y_pred_Run3, f1_Run3 = evaluate_version('Run3')

🚀 เริ่มรัน Pipeline Run 3...
⚙️ Step 1: เริ่มสกัด Features จากข้อมูล 500 เคส (ขั้นตอนนี้อาจใช้เวลา)...
   -> ดึงข้อมูลถึงเคสที่ 50/500...
   -> ดึงข้อมูลถึงเคสที่ 100/500...
   -> ดึงข้อมูลถึงเคสที่ 150/500...
   -> ดึงข้อมูลถึงเคสที่ 200/500...
   -> ดึงข้อมูลถึงเคสที่ 250/500...
   -> ดึงข้อมูลถึงเคสที่ 300/500...
   -> ดึงข้อมูลถึงเคสที่ 350/500...
   -> ดึงข้อมูลถึงเคสที่ 400/500...
   -> ดึงข้อมูลถึงเคสที่ 450/500...
   -> ดึงข้อมูลถึงเคสที่ 500/500...
✅ Step 1 เสร็จสิ้น! ได้ข้อมูลเตรียมเทรนทั้งหมด: 1712 ซี่
⚙️ Step 2: กำลัง Train โมเดล Random Forest...
Saved model to rf_classify_ml.pkl
✅ Step 2 เสร็จสิ้น! Train: 1370 ซี่ | Test: 342 ซี่
⚙️ Step 3: นำโมเดลไปทำนายผลทั้ง 500 เคส...
   -> ทำนายผล... 10% (50/500 เคส)
   -> ทำนายผล... 20% (100/500 เคส)
   -> ทำนายผล... 30% (150/500 เคส)
   -> ทำนายผล... 40% (200/500 เคส)
   -> ทำนายผล... 50% (250/500 เคส)
   -> ทำนายผล... 60% (300/500 เคส)
   -> ทำนายผล... 70% (350/500 เคส)
   -> ทำนายผล... 80% (400/500 เคส)
   -> ทำนายผล... 90% (450/5

# RUN 3: Random Forest classify_ml (Smart Fallback)

In [9]:
print("=== RF Model Status ===")
if "rf_model" in globals():
    print(f"rf_model     : {rf_model}")
    if rf_model is not None:
        print(f"classes_     : {rf_model.classes_}")
        print(f"n_estimators : {rf_model.n_estimators}")
        print(f"n_features   : {rf_model.n_features_in_}")
else:
    print("rf_model is not defined in this session.")

=== RF Model Status ===
rf_model     : RandomForestClassifier(class_weight='balanced', n_estimators=200,
                       random_state=42)
classes_     : ['Distal' 'Mesial' 'Occlusal']
n_estimators : 200
n_features   : 13


In [10]:
# =========================================================
# RUN 3: Random Forest classify_ml (Smart Fallback)
# =========================================================
import pandas as pd
import warnings
import json
import joblib
from pathlib import Path

# Ensure required globals exist when this cell runs standalone
if "FEATURE_COLS" not in globals():
    FEATURE_COLS = [
        "is_upper", "x_mean", "y_mean", "x_std", "y_std",
        "x_min", "x_max", "y_min", "x_range", "y_range",
        "x_centroid_dist", "aspect_ratio", "coverage",
    ]

if "RUN3_MODEL_PATH" not in globals():
    RUN3_MODEL_PATH = Path.cwd() / "rf_classify_ml.pkl"

if "rf_model" not in globals():
    rf_model = None

if "RUN3_OUTPUT_ROOT" not in globals():
    RUN3_OUTPUT_ROOT = Path.cwd() / "PCA_Output_Run3"
    RUN3_OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

if "RUN3_SEG_ROOT" not in globals() or "RUN3_CARIES_ROOT" not in globals():
    _SP_DIR = Path.cwd().parent
    RUN3_SEG_ROOT = _SP_DIR / "week2-Tooth Detection & Segmentation" / "500-segmentation+recognition"
    RUN3_CARIES_ROOT = _SP_DIR / "week3-Caries-to-Tooth Mapping" / "dental_analysis_output"

# ==========================================
# 0. โหลดโมเดลให้ชัวร์ก่อนรัน
# ==========================================
try:
    if RUN3_MODEL_PATH.exists():
        rf_model = joblib.load(RUN3_MODEL_PATH)
        print("✅ โหลดโมเดล rf_classify_ml.pkl สำเร็จ!", flush=True)
    else:
        print(f"❌ ไม่พบโมเดล: {RUN3_MODEL_PATH}", flush=True)
except Exception as e:
    print(f"❌ โหลดโมเดลไม่สำเร็จ: {e}", flush=True)
    rf_model = None

if "_extract_ml_feature_dict" not in globals():
    def _extract_ml_feature_dict(tid, tooth_pts, caries_pts):
        return None

# ==========================================
# 0. เตรียม Fallback Function (ดึง Baseline v4.5 มาใช้)
# ==========================================
if "classify_xthird" not in globals():
    if "classify_surface_v45" in globals():
        classify_xthird = classify_surface_v45
    else:
        def classify_xthird(tid, tooth_pts, caries_pts):
            return "Other", 0.0, {"method": "Fallback-Other"}

# ==========================================
# 1. อัปเดตฟังก์ชันทำนายผล (Step 1: Smart Fallback & Proba)
# ==========================================
def classify_ml(tid, tooth_pts, caries_pts):
    try:
        features = _extract_ml_feature_dict(tid, tooth_pts, caries_pts)
        if features is None or rf_model is None:
            return classify_xthird(tid, tooth_pts, caries_pts)

        feature_df = pd.DataFrame(
            [[features[col] for col in FEATURE_COLS]],
            columns=FEATURE_COLS
        )

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            proba = rf_model.predict_proba(feature_df)[0]

        classes = list(rf_model.classes_)
        valid = ["Occlusal", "Mesial", "Distal"]
        scores = {cls: proba[classes.index(cls)] for cls in valid if cls in classes}
        if not scores:
            return classify_xthird(tid, tooth_pts, caries_pts)
        prediction = max(scores, key=scores.get)

        return prediction, 0.0, {"method": "RandomForest_Proba"}
    except Exception:
        try:
            return classify_xthird(tid, tooth_pts, caries_pts)
        except Exception:
            return "Other", 0.0, {}

# ==========================================
# 2. สร้าง process_case_ml ถ้าไม่มี (Standalone Safety)
# ==========================================
if "_load_json_file" not in globals():
    def _load_json_file(path):
        if not path.exists():
            return None
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)

if "_load_run3_seg_case" not in globals():
    def _load_run3_seg_case(case_id):
        path = RUN3_SEG_ROOT / f"case {case_id}" / f"case_{case_id}_results.json"
        return _load_json_file(path)

if "_load_run3_caries_case" not in globals():
    def _load_run3_caries_case(case_id):
        path = RUN3_CARIES_ROOT / f"case {case_id}" / f"case_{case_id}_caries_mapping.json"
        return _load_json_file(path)

if "build_seg_map" not in globals():
    def build_seg_map(seg_data):
        return {
            str(t["tooth_id"]): t.get("pixel_coordinates", [])
            for t in seg_data.get("teeth_data", [])
        }

if "process_case_ml" not in globals():
    def process_case_ml(case_id, output_root):
        seg_data = _load_run3_seg_case(case_id)
        caries_data = _load_run3_caries_case(case_id)

        case_dir = output_root / f"case_{case_id}"
        case_dir.mkdir(parents=True, exist_ok=True)

        result = {"case_number": int(case_id), "teeth_data": []}

        if seg_data is None or caries_data is None:
            with open(case_dir / f"case_{case_id}.json", "w", encoding="utf-8") as f:
                json.dump(result, f, indent=2)
            return False, "Missing input data"

        seg_map = build_seg_map(seg_data)
        caries_list = caries_data.get("teeth_caries_data", [])

        for tooth in caries_list:
            tid = str(tooth.get("tooth_id", ""))
            tooth_pts = seg_map.get(tid, [])
            caries_pts = tooth.get("caries_coordinates", [])

            surface, angle, metadata = classify_ml(tid, tooth_pts, caries_pts)

            result["teeth_data"].append({
                "tooth_id": tid,
                "version": "Run3",
                "has_caries": True,
                "confidence": float(tooth.get("confidence", 0.0)),
                "caries_position_detail": surface,
                "predicted_surface_fine": surface,
                "tooth_coordinates": tooth_pts,
                "caries_coordinates": caries_pts,
            })

        with open(case_dir / f"case_{case_id}.json", "w", encoding="utf-8") as f:
            json.dump(result, f, indent=2)

        return True, f"OK ({len(result['teeth_data'])} teeth)"

# ==========================================
# 3. อัปเดต Config และลูปทำนายผลใหม่
# ==========================================
if "VERSION_CONFIGS" not in globals() or VERSION_CONFIGS is None:
    VERSION_CONFIGS = {}

VERSION_CONFIGS["Run3"] = {
    "out_dir": "PCA_Output_Run3",
    "classifier": classify_ml,
    "label": "Geometric Feature + ML (Smart Fallback)",
}

case_ids = list(range(1, 501))
total_cases = len(case_ids)
processed, failed = 0, 0

print("🚀 เริ่มรัน Step 1: Smart Fallback ทั้ง 500 เคส...", flush=True)
for i, case_id in enumerate(case_ids):
    result = process_case_ml(case_id, RUN3_OUTPUT_ROOT)
    ok = result[0] if isinstance(result, tuple) else bool(result)
    if ok:
        processed += 1
    else:
        failed += 1

    step = i + 1
    if step % 50 == 0 or step == total_cases:
        print(f"   -> ทำนายผล... {(step / total_cases) * 100:.0f}% ({step}/{total_cases} เคส)", flush=True)

print("🎉 ทำนายผลเสร็จสิ้น! กำลังรัน Evaluation...", flush=True)
if "evaluate_version" in globals():
    all_y_true_Run3, all_y_pred_Run3, f1_Run3 = evaluate_version("Run3")
else:
    print("evaluate_version is not available. Run the evaluation cell to compute metrics.")

✅ โหลดโมเดล rf_classify_ml.pkl สำเร็จ!
🚀 เริ่มรัน Step 1: Smart Fallback ทั้ง 500 เคส...
   -> ทำนายผล... 10% (50/500 เคส)
   -> ทำนายผล... 20% (100/500 เคส)
   -> ทำนายผล... 30% (150/500 เคส)
   -> ทำนายผล... 40% (200/500 เคส)
   -> ทำนายผล... 50% (250/500 เคส)
   -> ทำนายผล... 60% (300/500 เคส)
   -> ทำนายผล... 70% (350/500 เคส)
   -> ทำนายผล... 80% (400/500 เคส)
   -> ทำนายผล... 90% (450/500 เคส)
   -> ทำนายผล... 100% (500/500 เคส)
🎉 ทำนายผลเสร็จสิ้น! กำลังรัน Evaluation...

========== FINAL EVALUATION [Run3] ==========
Total Samples : 1979
Accuracy      : 0.8277
Precision     : 0.6437
Recall        : 0.6201
F1 Score      : 0.6317

Confusion Matrix:
          Occlusal  Mesial  Distal  Other
Occlusal       324      22      15     27
Mesial          26     538      91     15
Distal          27      84     776     34
Other            0       0       0      0

Classification Report:
              precision    recall  f1-score   support

    Occlusal       0.86      0.84      0.85       

## 4. Evaluation

parse XML ground truth + parse prediction refactor mapping + parser 
ทำ evaluation:
Accuracy
Precision
Recall
F1-score
Confusion Matrix

In [12]:
# =========================================================
# FULL EVALUATION + HYPOTHESIS TESTING (SELF-CONTAINED)
# =========================================================

from pathlib import Path
import json
import xml.etree.ElementTree as ET
import numpy as np
import pandas as pd

from sklearn.metrics import (
    confusion_matrix,
    classification_report,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

# =========================================================
# XML PARSER
# =========================================================
AIM_NS = "gme://caCORE.caCORE/4.4/edu.northwestern.radiology.AIM"
ISO_NS = "uri:iso.org:21090"
NS = {"aim": AIM_NS, "iso": ISO_NS}

SNODENT_SURFACE_MAP = {
    "144414D": "Occlusal",
    "146014D": "Distal",
    "145374D": "Mesial",
    "144474D": "Occlusal",
    "146074D": "Distal",
    "145434D": "Mesial",
}

DISPLAY_NAME_TO_SURFACE = {
    "Occlusal surface": "Occlusal",
    "Occlusal Surface": "Occlusal",
    "Distal Surface": "Distal",
    "Distal surface": "Distal",
    "Mesial Surface": "Mesial",
    "Mesial surface": "Mesial",
}

SNODENT_TO_FDI = {
    "161006D": "11", "160842D": "12", "160288D": "13", "161286D": "14",
    "160450D": "15", "160770D": "16", "161204D": "17", "160618D": "18",
    "160194D": "21", "160132D": "22", "160506D": "23", "161340D": "24",
    "160682D": "25", "161074D": "26", "160386D": "27", "160922D": "28",
    "161136D": "31", "160556D": "32", "160068D": "33", "160326D": "34",
    "161248D": "35", "160730D": "36", "161166D": "37", "160580D": "38",
    "160964D": "41", "160350D": "42", "160894D": "43", "160230D": "44",
    "161412D": "45", "160770D": "46", "161102D": "47", "160488D": "48",
}

VALID_SURFACES = ["Occlusal", "Mesial", "Distal", "Other"]


def _get_display_name(element):
    dn = element.find("iso:displayName", NS)
    return dn.get("value", "") if dn is not None else ""


def snodent_display_to_fdi(display_name):
    dn = display_name.lower()

    if "upper" in dn and "right" in dn:
        quadrant = 1
    elif "upper" in dn and "left" in dn:
        quadrant = 2
    elif "lower" in dn and "left" in dn:
        quadrant = 3
    elif "lower" in dn and "right" in dn:
        quadrant = 4
    else:
        return ""

    if "central incisor" in dn:
        pos = 1
    elif "lateral incisor" in dn:
        pos = 2
    elif "canine" in dn:
        pos = 3
    elif "first premolar" in dn:
        pos = 4
    elif "second premolar" in dn:
        pos = 5
    elif "first molar" in dn:
        pos = 6
    elif "second molar" in dn:
        pos = 7
    elif "third molar" in dn:
        pos = 8
    else:
        return ""

    return f"{quadrant}{pos}"


def parse_aim_xml(xml_path):
    try:
        tree = ET.parse(xml_path)
    except Exception:
        return None

    root = tree.getroot()
    anns = root.find("aim:imageAnnotations", NS)

    if anns is None:
        return None

    ann = anns.find("aim:ImageAnnotation", NS)

    if ann is None:
        return None

    tooth = ""
    surface = ""

    phys_coll = ann.find("aim:imagingPhysicalEntityCollection", NS)

    if phys_coll is not None:
        entity = phys_coll.find("aim:ImagingPhysicalEntity", NS)

        if entity is not None:
            char_coll = entity.find(
                "aim:imagingPhysicalEntityCharacteristicCollection", NS
            )

            if char_coll is not None:
                chars = char_coll.findall(
                    "aim:ImagingPhysicalEntityCharacteristic", NS
                )

                for ch in chars:
                    q_idx_el = ch.find("aim:questionIndex", NS)
                    q_idx = q_idx_el.get("value", "") if q_idx_el is not None else ""

                    tc = ch.find("aim:typeCode", NS)
                    if tc is None:
                        continue

                    code = tc.get("code", "")
                    display = _get_display_name(tc)

                    if q_idx == "0":
                        tooth = snodent_display_to_fdi(display)
                        if not tooth:
                            tooth = SNODENT_TO_FDI.get(code, "")

                    elif q_idx == "1":
                        surface = SNODENT_SURFACE_MAP.get(code, "")
                        if not surface:
                            surface = DISPLAY_NAME_TO_SURFACE.get(display, "")

    return {"tooth": tooth, "surface": surface}


# =========================================================
# LOAD FUNCTIONS
# =========================================================
def parse_case_ground_truth(case_folder):
    gt = []

    for xml_file in sorted(Path(case_folder).glob("*.xml")):
        parsed = parse_aim_xml(str(xml_file))

        if parsed is None:
            continue

        tooth = str(parsed.get("tooth", "Unknown"))
        surface = parsed.get("surface", "Other")

        if surface not in VALID_SURFACES:
            surface = "Other"

        gt.append({
            "tooth": tooth,
            "surface": surface
        })

    return gt


def load_prediction(case_num, out_dir):
    pred_path = Path(out_dir) / f"case_{case_num}" / f"case_{case_num}.json"

    if not pred_path.exists():
        return []

    with open(pred_path, "r") as f:
        data = json.load(f)

    preds = []

    for t in data.get("teeth_data", []):
        tooth = str(t.get("tooth_id", "Unknown"))

        surface = t.get(
            "predicted_surface_fine",
            t.get("caries_position_detail", "Other")
        )

        if surface not in VALID_SURFACES:
            surface = "Other"

        preds.append({
            "tooth": tooth,
            "surface": surface
        })

    return preds


def match_case(gt, pred):
    pred_dict = {p["tooth"]: p["surface"] for p in pred}

    y_true = []
    y_pred = []

    for g in gt:
        tooth = g["tooth"]
        gt_surface = g["surface"]
        pred_surface = pred_dict.get(tooth, "Other")

        y_true.append(gt_surface)
        y_pred.append(pred_surface)

    return y_true, y_pred


# =========================================================
# EVALUATION FUNCTION
# =========================================================
def evaluate_version(version):
    out_dir = f"PCA_Output_{version}"
    base_gt = Path("../data/500 cases with annotation")

    all_y_true = []
    all_y_pred = []

    for case_num in range(1, 501):
        gt_folder = base_gt / f"case {case_num}"

        gt = parse_case_ground_truth(gt_folder)
        pred = load_prediction(case_num, out_dir)

        if len(gt) == 0 and len(pred) == 0:
            continue

        yt, yp = match_case(gt, pred)

        all_y_true.extend(yt)
        all_y_pred.extend(yp)

    accuracy = accuracy_score(all_y_true, all_y_pred)
    precision = precision_score(all_y_true, all_y_pred, average="macro", zero_division=0)
    recall = recall_score(all_y_true, all_y_pred, average="macro", zero_division=0)
    f1 = f1_score(all_y_true, all_y_pred, average="macro", zero_division=0)

    cm = confusion_matrix(all_y_true, all_y_pred, labels=VALID_SURFACES)
    cm_df = pd.DataFrame(cm, index=VALID_SURFACES, columns=VALID_SURFACES)

    print(f"\n========== FINAL EVALUATION [{version}] ==========")
    print(f"Total Samples : {len(all_y_true)}")
    print(f"Accuracy      : {accuracy:.4f}")
    print(f"Precision     : {precision:.4f}")
    print(f"Recall        : {recall:.4f}")
    print(f"F1 Score      : {f1:.4f}")
    print("\nConfusion Matrix:")
    print(cm_df)

    print("\nClassification Report:")
    print(classification_report(
        all_y_true,
        all_y_pred,
        labels=VALID_SURFACES,
        zero_division=0
    ))

    return all_y_true, all_y_pred, f1


# =========================================================
# RUN EVALUATION BOTH VERSIONS
# =========================================================
# all_y_true_45, all_y_pred_45, f1_45 = evaluate_version("v4.5")
# all_y_true_46, all_y_pred_46, f1_46 = evaluate_version("v4.6")
all_y_true_45, all_y_pred_45, f1_45 = evaluate_version("Baseline")
# all_y_true_Run1, all_y_pred_Run1, f1_Run1 = evaluate_version("Run1")
# all_y_true_Run2, all_y_pred_Run2, f1_Run2 = evaluate_version("Run2")
all_y_true_Run3, all_y_pred_Run3, f1_Run3 = evaluate_version("Run3")

# ใช้ GT ของ v4.5 เป็นตัวหลัก
all_y_true = all_y_true_45

# =========================================================
# DEBUG: PRINT + SAVE ALL 3 LISTS
# =========================================================
print("\n========== LIST LENGTH CHECK ==========")
print("all_y_true    :", len(all_y_true))
# print("all_y_pred_45 :", len(all_y_pred_45))
# print("all_y_pred_46 :", len(all_y_pred_46))
print("all_y_pred_45 :", len(all_y_pred_45))
# print("all_y_pred_Run1 :", len(all_y_pred_Run1))
# print("all_y_pred_Run2 :", len(all_y_pred_Run2))
print("all_y_pred_Run3 :", len(all_y_pred_Run3))

print("\n========== FIRST 30 SAMPLES ==========")
for i in range(min(30, len(all_y_true))):
    print(
        f"{i+1:04d} | "
        f"GT={all_y_true[i]:10s} | "
        # f"v4.5={all_y_pred_45[i]:10s} | "
        # f"v4.6={all_y_pred_46[i]:10s}"
        f"v4.5={all_y_pred_45[i]:10s} | "
        # f"Run1={all_y_pred_Run1[i]:10s}"
        # f"Run2={all_y_pred_Run2[i]:10s}"
        f"Run3={all_y_pred_Run3[i]:10s}"
    )

# Save TXT files
Path("debug_all_y_true.txt").write_text(
    "\n".join(all_y_true),
    encoding="utf-8"
)

Path("debug_all_y_pred_45.txt").write_text(
    "\n".join(all_y_pred_45),
    encoding="utf-8"
)

# Path("debug_all_y_pred_46.txt").write_text(
#     "\n".join(all_y_pred_46),
#     encoding="utf-8"

Path ("debug_all_y_pred_Run3.txt").write_text(
    "\n".join(all_y_pred_Run3),
    encoding="utf-8"
)




========== FINAL EVALUATION [Baseline] ==========
Total Samples : 1979
Accuracy      : 0.7049
Precision     : 0.5454
Recall        : 0.4638
F1 Score      : 0.4705

Confusion Matrix:
          Occlusal  Mesial  Distal  Other
Occlusal        77     121     153     37
Mesial           5     555      80     30
Distal          31      80     763     47
Other            0       0       0      0

Classification Report:
              precision    recall  f1-score   support

    Occlusal       0.68      0.20      0.31       388
      Mesial       0.73      0.83      0.78       670
      Distal       0.77      0.83      0.80       921
       Other       0.00      0.00      0.00         0

    accuracy                           0.70      1979
   macro avg       0.55      0.46      0.47      1979
weighted avg       0.74      0.70      0.69      1979


========== FINAL EVALUATION [Run3] ==========
Total Samples : 1979
Accuracy      : 0.8277
Precision     : 0.6437
Recall        : 0.6201
F1 Score   

14530